## Getting Started with ML project With Mlflow

- installing MLFLOW
- starting a local mlflow tracking server
- Logging and registering a model with Mlflow 
- loading a logged model for inference using MLflow pyfunc flavour
- viewing the experiment results in the Mlflow UI


In [2]:
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import mlflow
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score
import mlflow.sklearn 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing)

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

In [3]:
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [6]:
#preparing the dataset 

data=pd.DataFrame(housing.data, columns=housing.feature_names)
data['Price']=housing.target
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
data.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


### Train test split Model Hyperparameter Tuning MLFLOW Experiments

In [8]:
from urllib.parse import urlparse
## dividing data in independance and dependance features
X=data.drop(columns=['Price'])
y=data['Price']


In [11]:
## Hyperparameter tuning using grid search cv

def hyperparameter_tuning(X_train, y_train,param_grid):
    from sklearn.model_selection import GridSearchCV
    rf=RandomForestRegressor()
    
    grid_search=GridSearchCV(estimator=rf,param_grid=param_grid,
                             cv=3, n_jobs=-1, verbose=2, scoring="neg_mean_squared_error")
    grid_search.fit(X_train, y_train)
    return grid_search.best_params_

In [12]:
## split data into training and test sets
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.20, random_state=42)

from mlflow.models import infer_signature
infer_signature(X_train, y_train)

# defining hyperparameter grid
param_grid={ 
    'n_estimators':[100,200], 
    'max_depth':[6,12,None], 
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2],
    'random_state':[42]}


#Start Mlflow experiments

with mlflow.start_run():

    grid_search=hyperparameter_tuning(X_train,y_train,param_grid)

    ## get the best model
    best_model=grid_search.best_estimator_
    
    # Evaluate best model
    y_pred = best_model.predict(X_test)
    mse=mean_squared_error(y_test, y_pred)

    ##Loag best parameters and metrics

    mlflow.log_param("best_n_estimators", grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf", grid_search.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse", mse)

    # tracking url

    mlflow.set_tracking_uri(uri="http://localhost:5000")
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="HousePricePredictionModel")
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=signature)

    print(f"Best Model Parameters: {grid_search.best_params_}")
    print(f"mean squared error: {mse}")

2025/12/25 16:22:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/25 16:22:07 INFO mlflow.store.db.utils: Updating database tables
2025/12/25 16:22:07 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/25 16:22:07 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/25 16:22:07 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/25 16:22:07 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Fitting 3 folds for each of 24 candidates, totalling 72 fits


AttributeError: 'dict' object has no attribute 'best_estimator_'

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mlflow.models import infer_signature
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn

# split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# infer model signature (FIX: store it)
signature = infer_signature(X_train, y_train)

# defining hyperparameter grid
param_grid = { 
    'n_estimators': [100, 200], 
    'max_depth': [6, 12, None], 
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'random_state': [42]
}

# FIX: set tracking URI BEFORE starting run
mlflow.set_tracking_uri("http://localhost:5000")

# Start MLflow experiment
with mlflow.start_run():

    grid_search = hyperparameter_tuning(X_train, y_train, param_grid)

    # get the best model
    best_model = grid_search.best_estimator_
    
    # Evaluate best model
    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    # Log best parameters
    mlflow.log_param("best_n_estimators", grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf", grid_search.best_params_['min_samples_leaf'])

    # Log metric
    mlflow.log_metric("mse", mse)

    # Check tracking store type
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="HousePricePredictionModel"
        )
    else:
        mlflow.sklearn.log_model(
            best_model,
            "model",
            signature=signature
        )

    print(f"Best Model Parameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")


2025/12/25 16:37:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'HousePricePredictionModel'.
2025/12/25 16:38:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HousePricePredictionModel, version 1


Best Model Parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200, 'random_state': 42}
Mean Squared Error: 0.2540345930813259
🏃 View run victorious-stoat-818 at: http://localhost:5000/#/experiments/0/runs/75b3cbda9eb24625b706d00e75b4cb3d
🧪 View experiment at: http://localhost:5000/#/experiments/0


Created version '1' of model 'HousePricePredictionModel'.
